In [14]:
import numpy as numpy
import pandas as pd

# for api
import json
import requests

In [15]:
def get(item):
    response = requests.get('https://fantasy.premierleague.com/api/'+item+'/')
    data = response.json()
    return data

In [16]:
# Past data

In [17]:
# creates one df which has each players team and position
def get_general_player_df(general_info):
    # player info
    players = pd.json_normalize(general_info['elements'])
    player_info = players[['id', 'web_name', 'first_name', 'second_name', 'team', 'element_type']].copy()

    # teams info
    teams = pd.json_normalize(general_info['teams'])
    teams = teams[['id', 'name']].copy()

    # positions info
    positions = pd.json_normalize(general_info['element_types'])
    positions = positions[['id', 'plural_name_short']].copy()

    # combining
    general_player_info = pd.merge(player_info, positions, left_on='element_type', right_on='id', how='left')
    general_player_info = pd.merge(player_info, teams, left_on='team', right_on='id')

    # editing columns
    general_player_info = general_player_info.drop(columns=['team', 'element_type', 'id_y'])
    general_player_info = general_player_info.rename(columns={'id_x':'id', 'name':'team'})

    return general_player_info

In [18]:
# for a given player this calcualtes all the past variables
def individual_player_predictor_df(player, general_player_df, teams, decay_span, rolling_window):
    # player is just the players id
    # general_player_df just contains their name and their team
    first_name = general_player_df.loc[general_player_df['id']==player, 'first_name'].item()
    second_name = general_player_df.loc[general_player_df['id']==player, 'second_name'].item()
    name = f'{first_name} {second_name}'
    
    #### get gw info for each player
    # this brings me up info of the player
    player_df = get(f'element-summary/{player}')
    # this gets the past part of the player df, remeber the keys for 'get(f'element-summary/{player}')' are 'fixtures', 
    #'hisotry' (this seaon psat gw), and 'history past' (previous season data)
    player_df = pd.json_normalize(player_df['history'])
    # columns for history include ['element', 'fixture', 'opponent_team', 'total_points', 'was_home',
        #'kickoff_time', 'team_h_score', 'team_a_score', 'round', 'modified',
       #'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded',
       #'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards',
       #'red_cards', 'saves', 'bonus', 'bps', 'influence', 'creativity',
       #'threat', 'ict_index', 'clearances_blocks_interceptions', 'recoveries',
       #'tackles', 'defensive_contribution', 'starts', 'expected_goals',
       #'expected_assists', 'expected_goal_involvements',
       #'expected_goals_conceded', 'value', 'transfers_balance', 'selected',
       #'transfers_in', 'transfers_out']
    player_df = player_df[['element', 'total_points', 'influence', 'creativity', 'threat']]

    # combine past gw info with general info (ie combine name and team with past gw df)
    final_df = pd.merge(player_df, general_player_df, left_on='element', right_on='id')
    # editing opponent team from id to name - this is y we need teams df
    player_df = final_df


    # make extra variables
    # Calculate EWMA and rolling averages
    player_df['ewma_total_points'] = player_df['total_points'].shift(1).ewm(span=decay_span, adjust=False).mean().round(2)
    player_df['ewma_influence'] = player_df['influence'].shift(1).ewm(span=decay_span, adjust=False).mean().round(2)
    player_df['ewma_creativity'] = player_df['creativity'].shift(1).ewm(span=decay_span, adjust=False).mean().round(2)
    player_df['ewma_threat'] = player_df['threat'].shift(1).ewm(span=decay_span, adjust=False).mean().round(2)

    player_df['avg_total_points_20'] = player_df['total_points'].shift(1).rolling(window=rolling_window, min_periods=1).mean().round(2)
    player_df['avg_influence_20'] = player_df['influence'].shift(1).rolling(window=rolling_window, min_periods=1).mean().round(2)
    player_df['avg_creativity_20'] = player_df['creativity'].shift(1).rolling(window=rolling_window, min_periods=1).mean().round(2)
    player_df['avg_threat_20'] = player_df['threat'].shift(1).rolling(window=rolling_window, min_periods=1).mean().round(2)

    player_df['Full name'] = name
    # remove non-variable colums
    cols = ['id', 'Full name',
            'ewma_total_points', 'ewma_influence', 'ewma_creativity', 'ewma_threat',
            'avg_total_points_20', 'avg_influence_20', 'avg_creativity_20', 'avg_threat_20']

    player_df = player_df[cols]
    # return only last weekl
    final_week = player_df.iloc[[-1]].copy()

    return final_week

In [19]:
# This applies the prevuous function to every player, outputting the past variables for every player
def get_player_predictor_variables_df():
    decay_span = 5
    rolling_window = 20

    general_info = get('bootstrap-static')
    # get general name and team info
    general_player_df = get_general_player_df(general_info)
    teams = pd.json_normalize(general_info['teams'])[['id', 'name']]

    # list of player
    players = general_player_df['id']
    all_player_dfs = []

    # iterate through each player to get the past variables for the next fixture
    for player in players:
        past_variables = individual_player_predictor_df(player, general_player_df, teams, decay_span, rolling_window)
        all_player_dfs.append(past_variables)

    all_player_df = pd.concat(all_player_dfs, ignore_index=True)

    return all_player_df


In [20]:
# Present Variables

In [21]:
def get_present_variables():
    ### presnet info - current team and value
    general_info = get("bootstrap-static")
    
    players = pd.DataFrame(general_info["elements"])
    teams = pd.DataFrame(general_info["teams"])[["id", "short_name"]]
    
    # Convert price to £ format
    players["value"] = players["now_cost"] / 10
    
    # Map team ID → team name
    id_to_team = dict(zip(teams["id"], teams["short_name"]))
    players["team_name"] = players["team"].map(id_to_team)
    
    # Keep only what you need
    current_info = players[[
        "id",
        "web_name",
        "team_name",
        "value"
    ]]

    return current_info

In [22]:
# Team rankings

In [23]:
team_rank_2024_25_FLIPPED = {
    # Promoted (strongest of promoted gets higher than playoff winner)
    "SUN": 20,  # Sunderland (playoffs)
    "BUR": 19,  # Burnley (2nd)
    "LEE": 18,  # Leeds (1st)

    # 2024/25 PL teams (flipped so 1st = 1)
    "TOT": 17,
    "WOL": 16,
    "MUN": 15,
    "WHU": 14,
    "EVE": 13,
    "CRY": 12,
    "FUL": 11,
    "BRE": 10,
    "BOU": 9,
    "BHA": 8,
    "NFO": 7,
    "AVL": 6,
    "NEW": 5,
    "CHE": 4,
    "MCI": 3,
    "ARS": 2,
    "LIV": 1,
}

In [24]:
def get_future_variables():

    # --- your ranking dictionary must exist in scope ---
    # team_rank_2024_25_FLIPPED = {...}

    # Fixtures (future only)
    fixtures = pd.DataFrame(get("fixtures"))[["event", "team_h", "team_a", "finished"]]
    future = fixtures[fixtures["finished"] == False].copy()

    # Team-centric rows + venue
    home = (
        future.rename(columns={"team_h": "team", "team_a": "opponent"})[["event", "team", "opponent"]]
        .assign(is_home=1)
    )

    away = (
        future.rename(columns={"team_a": "team", "team_h": "opponent"})[["event", "team", "opponent"]]
        .assign(is_home=0)
    )

    team_fixtures = pd.concat([home, away], ignore_index=True).sort_values("event", na_position="last")

    # Players + teams
    bootstrap = get("bootstrap-static")
    players = pd.DataFrame(bootstrap["elements"])[["id", "web_name", "team"]]
    teams = pd.DataFrame(bootstrap["teams"])[["id", "short_name"]]
    id_to_team = dict(zip(teams["id"], teams["short_name"]))

    # Merge fixtures onto players
    players_fixtures = players.merge(team_fixtures, on="team", how="left")
    players_fixtures["Opponent"] = players_fixtures["opponent"].map(id_to_team)
    players_fixtures["event"] = players_fixtures["event"].astype("Int64")

    # Map opponent short_name -> opponent rank
    players_fixtures["opp_rank"] = players_fixtures["Opponent"].map(team_rank_2024_25_FLIPPED)

    # Keep tidy
    final_fixtures = players_fixtures[["id", "web_name", "event", "opp_rank", "is_home"]].copy()

    # Sort within player, create fixture number 1..8, keep first 8
    final_fixtures = final_fixtures.sort_values(["web_name", "event"], na_position="last")
    final_fixtures["fixture_no"] = final_fixtures.groupby("web_name").cumcount() + 1
    final_fixtures = final_fixtures[final_fixtures["fixture_no"] <= 8]

    # Pivot to wide: opp_rank columns
    opp = (
        final_fixtures.pivot(index="web_name", columns="fixture_no", values="opp_rank")
        .rename(columns=lambda k: f"opp_rank_{k}")
        .reset_index()
    )

    # Pivot to wide: is_home columns
    homeaway = (
        final_fixtures.pivot(index="web_name", columns="fixture_no", values="is_home")
        .rename(columns=lambda k: f"is_home_{k}")
        .reset_index()
    )

    # Merge pivots + keep id too (one id per web_name)
    ids = players[["web_name", "id"]].drop_duplicates()
    df = ids.merge(opp, on="web_name", how="left").merge(homeaway, on="web_name", how="left")

    return df

In [25]:
future_variables = get_future_variables()

In [26]:
past_variables_df = get_player_predictor_variables_df()
present_variables = get_present_variables()
future_variables = get_future_variables()

In [27]:
all_variables = pd.merge(past_variables_df, present_variables, on="id")
all_variables = pd.merge(all_variables, future_variables, on="id")
all_variables = all_variables.drop(columns=['web_name_y'])
all_variables.head(20)
all_variables.columns

Index(['id', 'Full name', 'ewma_total_points', 'ewma_influence',
       'ewma_creativity', 'ewma_threat', 'avg_total_points_20',
       'avg_influence_20', 'avg_creativity_20', 'avg_threat_20', 'web_name_x',
       'team_name', 'value', 'opp_rank_1', 'opp_rank_2', 'opp_rank_3',
       'is_home_1', 'is_home_2', 'is_home_3'],
      dtype='object')

In [28]:
all_variables.head(20)

,id,Full name,ewma_total_points,ewma_influence,ewma_creativity,ewma_threat,avg_total_points_20,avg_influence_20,avg_creativity_20,avg_threat_20,web_name_x,team_name,value,opp_rank_1,opp_rank_2,opp_rank_3,is_home_1,is_home_2,is_home_3
0,1,David Raya Martín,6.45,13.24,0.07,0.00,4.35,13.74,1.16,0.00,Raya,ARS,6.2,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Kepa Arrizabalaga Revuelta,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,Arrizabalaga,ARS,4.0,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Karl Hein,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,Hein,ARS,4.0,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Tommy Setford,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,Setford,ARS,3.9,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Gabriel dos Santos Magalhães,6.99,19.57,2.66,13.95,6.35,24.75,3.28,10.50,Gabriel,ARS,7.3,NaN,NaN,NaN,NaN,NaN,NaN
5,6,William Saliba,6.12,20.28,6.80,3.31,4.40,16.70,7.94,3.50,Saliba,ARS,6.3,NaN,NaN,NaN,NaN,NaN,NaN
6,7,Riccardo Calafiori,4.00,8.20,13.83,11.69,1.55,2.99,3.52,4.50,Calafiori,ARS,5.6,NaN,NaN,NaN,NaN,NaN,NaN
7,8,Jurriën Timber,0.46,1.37,1.11,0.22,3.15,9.39,8.68,3.75,J.Timber,ARS,6.0,NaN,NaN,NaN,NaN,NaN,NaN
8,9,Jakub Kiwior,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,Kiwior,ARS,5.4,NaN,NaN,NaN,NaN,NaN,NaN
9,10,Myles Lewis-Skelly,2.70,4.29,3.70,1.05,0.90,2.46,1.81,0.75,Lewis-Skelly,ARS,5.0,NaN,NaN,NaN,NaN,NaN,NaN
